# Concrete and wine forest optimization

*This notebook requires a full Gurobi license and internet access for the
first OpenML download; later runs use the local cache. For a version that fits
a size-limited license, see
[Concrete Strength Optimization with Tree Ensembles](https://gurobi-optimization-gurobi-machine-learning.readthedocs-hosted.com/en/latest/auto_examples/example5_concrete_strength.html).*

We first maximize a random forest's prediction of concrete compressive
strength: load the data, train a forest, formulate the optimization model,
solve it with the `"misic"` formulation, and inspect the resulting features.
We then repeat the workflow for red-wine quality. Only the final section
compares formulations on these forests.

The applications revisit Mišić, *Optimization of Tree Ensembles*, Operations
Research 68(5):1605–1624, 2020
([doi:10.1287/opre.2019.1928](https://doi.org/10.1287/opre.2019.1928)).
This is an adaptation, not an exact reproduction: we use scikit-learn rather
than R's `randomForest`. These are surrogate-optimization examples, not
validated design recommendations.

This notebook is self-contained.

In [ ]:
import time

import gurobipy as gp
import pandas as pd
from IPython.display import display
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestRegressor

from gurobi_ml import add_predictor_constr

pd.set_option("display.float_format", "{:.3f}".format)

## Load concrete strength data and train the predictor

The UCI concrete dataset (Yeh, 1998) records ingredient quantities and curing
age, with compressive strength in MPa as the target. OpenML downloads and
caches it; download errors are not suppressed.

We use 20 trees of depth at most 5 to keep the example manageable. As in the
paper, each split considers a third of the features and each leaf contains
at least five training samples. The paper uses unlimited depth; our smaller
forests illustrate the workflow rather than reproduce its large experiments.

In [ ]:
def train_forest(X, y):
    return RandomForestRegressor(
        n_estimators=20,
        max_depth=5,
        max_features=1 / 3,
        min_samples_leaf=5,
        random_state=0,
    ).fit(X.to_numpy(), y.to_numpy())


frame = fetch_openml(data_id=4353, as_frame=True).frame
X_concrete = frame.iloc[:, :-1].astype(float)
y_concrete = frame.iloc[:, -1].astype(float)
concrete_forest = train_forest(X_concrete, y_concrete)
X_concrete.head()

## Model prediction maximization with one formulation

Let $f(x)$ be the trained forest's prediction. We choose the input vector $x$:

$$
\max_x f(x) \qquad
\text{subject to }\min_i X_{ij} \le x_j \le \max_i X_{ij}
\quad\text{for each feature }j.
$$

The bounds are the observed feature ranges. `add_predictor_constr` creates the
forest's prediction variable and constraints, and we use that variable as the
objective.

We use `"misic"`, the formulation from the paper, whose split decisions are
shared across trees, with the default `epsilon=0`. The solution section below
discusses what this means for the result.

In [ ]:
def solve(predictor, X, formulation="misic"):
    """Build and solve the model; return a dict of results and the solution."""
    start = time.perf_counter()
    model = gp.Model()
    model.Params.OutputFlag = 0
    model.Params.Seed = 0
    x = model.addMVar(
        (1, X.shape[1]),
        lb=X.min().to_numpy(),
        ub=X.max().to_numpy(),
        name="features",
    )
    # epsilon=0 lets the solver put features exactly on split thresholds and take
    # either branch there, so the objective can exceed what the forest predicts
    # at x (see "prediction at x" in the results). A positive epsilon removes
    # this ambiguity, but see the discussion after the first solve.
    prediction = add_predictor_constr(
        model, predictor, x, formulation=formulation, epsilon=0.0
    )
    model.setObjective(prediction.output.sum(), gp.GRB.MAXIMIZE)
    build_time = time.perf_counter() - start
    model.optimize()

    # Without an incumbent, only the status and the bound are available.
    solved = model.SolCount > 0
    nan = float("nan")
    result = {
        "formulation": formulation,
        "variables": model.NumVars,
        "binaries": model.NumBinVars,
        "linear constraints": model.NumConstrs,
        "general constraints": model.NumGenConstrs,
        "build [s]": build_time,
        "solve [s]": model.Runtime,
        "status": model.Status,
        "objective": model.ObjVal if solved else nan,
        "bound": model.ObjBound,
        "gap": model.MIPGap if solved else nan,
        "prediction at x": predictor.predict(x.X)[0] if solved else nan,
    }
    solution = pd.Series(x.X[0], index=X.columns, name="features") if solved else None
    return result, solution

## Solve the concrete model

`solve` builds and solves the model with a fixed solver seed. It returns plain
result data: if the solve stops early, `status`, `bound` and `gap` show
whether the incumbent is proved optimal. A solve without an incumbent has no
objective and no solution.

In [ ]:
concrete_result, concrete_solution = solve(concrete_forest, X_concrete)
pd.Series(concrete_result)

## Inspect the concrete solution

The table gives the ingredient quantities and curing age chosen by the model.
The objective is the predicted strength in MPa.

In [ ]:
concrete_solution

The objective is not a measurement, and independent feature bounds do not
describe a feasible recipe or production process.

**The objective is optimistic.** With `epsilon=0`, an input exactly on a split
threshold satisfies both branches, and the solver is free to place features on
thresholds and take the better branch. The forest's `predict` must choose one
branch, and it rounds inputs to single precision, so `prediction at x` can be
noticeably lower than the `objective` (compare them above). Even with shared
split decisions, reading the raw solver input is not an exactness check.

A positive `epsilon` removes the ambiguity, but it must be larger than
Gurobi's `IntFeasTol` (1e-5 by default) and than the single-precision
resolution of the feature, otherwise `predict` rounds the input back onto the
threshold. It can also make the model infeasible. Reading the chosen leaves
from `tree_leaves` avoids recomputing the prediction altogether.

See [Tree Ensemble Formulations](Tree%20Ensemble%20Formulations.ipynb) for
selected-leaf checks and the effect of `epsilon`.

## Apply the same model to red-wine quality

The UCI red-wine dataset (Cortez et al., 2009) contains physicochemical
measurements and a quality score. We train another 20-tree forest with the
same settings and solve the same bounded prediction-maximization model with
`"misic"`. The objective is now a predicted quality score rather than
strength in MPa, and each measurement is bounded by its observed range.

In [ ]:
# Quality labels are numeric, even though OpenML marks them categorical.
data = fetch_openml(data_id=40691, as_frame=True, read_csv_kwargs={"dtype": float})
X_wine = data.data.astype(float)
y_wine = data.target.astype(float)
wine_forest = train_forest(X_wine, y_wine)

## Solve and inspect the wine model

We reuse `solve`. The first table gives the solve status and objective, the
second the chosen measurements.

In [ ]:
wine_result, wine_solution = solve(wine_forest, X_wine)
display(pd.Series(wine_result))
display(wine_solution)

As with concrete, the result is a prediction at an input within the bounds.
Because the bounds are the observed ranges of each measurement separately,
the optimizer may combine extreme values that never occur together in real
wines. Compare the chosen measurements with the dataset's ranges before
reading anything into them.

## Compare formulations

The `formulation` argument of `add_predictor_constr` selects among five tree
ensemble formulations: `"leaf"` (the default, which models each tree on its
own), `"misic"` (Mišić's formulation, with split variables shared across
trees), `"misic_lazy"` (`"misic"` with lazy split constraints), `"ocean"`
(Parmentier and Vidal's) and `"biggs_perakis"` (Biggs and Perakis's
projected formulation). They represent the same function and differ in size
and strength; see [Decision Trees and Tree Ensembles](https://gurobi-optimization-gurobi-machine-learning.readthedocs-hosted.com/en/latest/user/mip-models.html#decision-trees-and-tree-ensembles) in the user
guide.

We reuse the two trained forests. Only the formulation changes; the data,
objective, feature bounds and solver seed stay fixed. The table reports the
size of each model, the time to build it and to solve it, and the solution
status, objective, bound and gap.

**Runtime:** these small forests typically solve in a few seconds, but the
solves have no time limit. Larger forests or deeper trees can take
substantially longer, and a robust timing study also needs multiple seeds.

In [ ]:
FORMULATIONS = ["leaf", "misic", "misic_lazy", "ocean", "biggs_perakis"]

rows = []
for dataset, forest, X in [
    ("concrete", concrete_forest, X_concrete),
    ("winequality-red", wine_forest, X_wine),
]:
    for formulation in FORMULATIONS:
        result, _ = solve(forest, X, formulation)
        rows.append({"dataset": dataset, **result})
pd.DataFrame(rows)

Inspect `status`, `objective`, `bound`, and `gap` before comparing solve times,
and compare `prediction at x` with `objective`.
A time-limited run is not a proved optimum; a missing objective means no
incumbent was found. With `epsilon=0`, `"leaf"`, `"ocean"` and
`"biggs_perakis"` can select incompatible leaves across trees, so objective
differences need not be caused by solver progress alone.

Rankings depend on the ensemble, surrounding constraints, hardware, Gurobi
version, threads, and seed. These results are not a substitute for testing
your actual decision problem.

For an application that adds constraints on selected tree leaves, see
[Drug Design with Tree Ensembles](../tree-ensembles-merck/Drug%20Design%20with%20Tree%20Ensembles.ipynb).

Copyright © 2023-2026 Gurobi Optimization, LLC